# Zadanie 4 — Whitespot Analysis

## 4.0 Importy i dane

In [1]:
import json, pickle, os, warnings
import pandas as pd
import numpy as np
import geopandas as gpd
import folium
import folium.plugins as fplug
from shapely.geometry import Point
from shapely import wkt
from sklearn.neighbors import BallTree
import snowflake.connector
warnings.filterwarnings('ignore')

with open('../credentials/credentials.json') as f:
    creds = json.load(f)

with open('../data/processed/model.pkl', 'rb') as f:
    saved = pickle.load(f)

pipeline         = saved['pipeline']          # pełny pipeline (SelectKBest+scaler+est)
slim_pipeline    = saved['slim_pipeline']     # tylko scaler+est, przyjmuje selected_features
feature_cols     = saved['feature_cols']      # wszystkie 93 cechy (do padding zera)
log_cols         = saved['log_features']      # cechy do log1p PRZED pipeline (lista nazw)
selected_features = saved['selected_features']  # 12 cech faktycznie używanych
log_selected     = saved['log_selected']      # podzbiór selected_features wymagający log1p
log_target       = saved['log_target']

locations   = pd.read_csv('../dane/ds_locations.csv')
competitors = pd.read_csv('../dane/ds_competitors.csv')
area_df     = pd.read_csv('../dane/ds_analysis_area.csv')
districts_df = pd.read_csv('../dane/ds_districts.csv')

buildings  = pd.read_csv('../data/raw/buildings.csv.gz',  compression='gzip')
population = pd.read_csv('../data/raw/population.csv.gz', compression='gzip')

area_poly = wkt.loads(area_df['geometry'].iloc[0])
lng_min, lat_min, lng_max, lat_max = area_poly.bounds

print(f'Model: {saved["best_name"]}  k={saved["best_k"]}')
print(f'CV R² = {saved["cv_R2_mean"]:.3f} ± {saved["cv_R2_std"]:.3f}')
print(f'CV RMSE = {saved["cv_RMSE_PLN"]:,.0f} PLN')
print(f'Selected features ({len(selected_features)}): {selected_features}')
print(f'Log-transform before pipeline: {log_selected}')


Model: Ridge  k=7
CV R² = 0.216 ± 0.043
CV RMSE = 90,374 PLN
Selected features (7): ['active_days_2000m', 'commercial_area_3000m', 'public_area_3000m', 'commercial_count_3000m', 'commercial_area_share_3000m', 'public_area_share_3000m', 'building_floor_ratio_3000m']
Log-transform before pipeline: []


## 4.1 Siatka punktów

In [2]:
step = 0.004  # ~400 m kompromis gęstość/czas
xs = np.arange(lng_min, lng_max, step)
ys = np.arange(lat_min, lat_max, step)

grid_points = []
for x in xs:
    for y in ys:
        if area_poly.contains(Point(x, y)):
            grid_points.append({'lat': y, 'lng': x})

grid_df = pd.DataFrame(grid_points)
grid_df['location_id'] = [f'GRID_{i:05d}' for i in range(len(grid_df))]
print(f'Siatka: {len(grid_df):,} punktów')


Siatka: 43,811 punktów


## 4.2 GeoDataFrames i bufory

In [3]:
grid_gdf = gpd.GeoDataFrame(
    grid_df,
    geometry=gpd.points_from_xy(grid_df.lng, grid_df.lat),
    crs=4326
).to_crs(2180)

grid_buf1000 = grid_gdf[['location_id','geometry']].copy()
grid_buf1000['geometry'] = grid_buf1000.geometry.buffer(1000)

grid_buf3000 = grid_gdf[['location_id','geometry']].copy()
grid_buf3000['geometry'] = grid_buf3000.geometry.buffer(3000)

print('Bufory 1000 m i 3000 m gotowe.')


Bufory 1000 m i 3000 m gotowe.


## 4.3 Cechy budynków — 1000 m i 3000 m

In [4]:
RESIDENTIAL = {
    'budynkiMieszkalneJednorodzinne','budynkiODwochMieszkaniach',
    'budynkiOTrzechIWiecejMieszkaniach','budynkiZbiorowegoZamieszkania',
}
COMMERCIAL = {
    'budynkiHandlowoUslugowe','budynkiBiurowe','budynkiHoteli',
    'ogolnodostepneObiektyKulturalne','budynkiLacznosciDworcowITerminali',
}
INDUSTRIAL = {'budynkiPrzemyslowe','zbiornikSilosIBudynkiMagazynowe'}
PUBLIC = {
    'budynkiSzkolIInstytucjiBadawczych','budynkiSzpitaliIZakladowOpiekiMedycznej',
    'budynkiKultuReligijnego','budynkiKulturyFizycznej','budynkiMuzeowIBibliotek',
}

print('Parsowanie budynków...')
buildings_gdf = gpd.GeoDataFrame(
    buildings,
    geometry=gpd.GeoSeries.from_wkt(buildings['geometry'], crs=4326),
    crs=4326
).to_crs(2180)[['teryt','funogolnabudynku_desc','area','geometry']].copy()

for cls, cats in [('residential',RESIDENTIAL),('commercial',COMMERCIAL),
                  ('industrial',INDUSTRIAL),('public',PUBLIC)]:
    mask = buildings_gdf['funogolnabudynku_desc'].isin(cats)
    buildings_gdf[f'is_{cls}']   = mask
    buildings_gdf[f'area_{cls}'] = buildings_gdf['area'] * mask


def building_features_for_radius(buf_gdf, radius_m):
    joined = gpd.sjoin(buildings_gdf, buf_gdf.to_crs(2180), how='inner', predicate='intersects')
    buf_area_m2 = np.pi * radius_m ** 2
    agg = joined.groupby('location_id').agg(
        buildings_count   =('teryt',          'count'),
        total_building_area=('area',          'sum'),
        residential_area  =('area_residential','sum'),
        commercial_area   =('area_commercial', 'sum'),
        industrial_area   =('area_industrial', 'sum'),
        public_area       =('area_public',     'sum'),
        commercial_count  =('is_commercial',   'sum'),
    ).reset_index()
    total = agg['total_building_area'].replace(0, np.nan)
    for cls in ['residential','commercial','industrial','public']:
        agg[f'{cls}_area_share'] = (agg[f'{cls}_area'] / total).fillna(0)
    agg['building_floor_ratio']      = agg['total_building_area'] / buf_area_m2
    agg['buildings_density_per_km2'] = agg['buildings_count'] / (buf_area_m2 / 1e6)
    rename = {c: f'{c}_{radius_m}m' for c in agg.columns if c != 'location_id'}
    return agg.rename(columns=rename)

print('Spatial join budynki × buf1000...')
bf_1000 = building_features_for_radius(grid_buf1000, 1000)
print('Spatial join budynki × buf3000...')
bf_3000 = building_features_for_radius(grid_buf3000, 3000)
building_features = bf_1000.merge(bf_3000, on='location_id', how='outer')
print(f'Building features: {len(building_features):,} × {building_features.shape[1]-1}')


Parsowanie budynków...


Spatial join budynki × buf1000...


Spatial join budynki × buf3000...


Building features: 43,811 × 26


## 4.4 Cechy demograficzne — 3000 m

In [5]:
pop_gdf = gpd.GeoDataFrame(
    population,
    geometry=gpd.points_from_xy(population.lng, population.lat),
    crs=4326
).to_crs(2180)
pop_gdf = pop_gdf[(pop_gdf['household_size'] > 0) & (pop_gdf['household_size'] <= 10)].copy()

print('Spatial join populacja × buf3000...')
joined_p = gpd.sjoin(
    pop_gdf[['total','households','household_size','geometry']],
    grid_buf3000, how='inner', predicate='within'
)
buf_area_km2_3000 = np.pi * 3000**2 / 1e6

pop_features = joined_p.groupby('location_id').apply(
    lambda x: pd.Series({
        'population_total_3000m':   x['total'].sum(),
        'households_count_3000m':   x['households'].sum(),
        'avg_household_size_3000m': (
            (x['household_size'] * x['households']).sum() / x['households'].sum()
            if x['households'].sum() > 0 else np.nan
        ),
        'population_per_km2_3000m': x['total'].sum() / buf_area_km2_3000,
        'households_per_km2_3000m': x['households'].sum() / buf_area_km2_3000,
    })
).reset_index()
print(f'Population features: {len(pop_features):,} × {pop_features.shape[1]-1}')


Spatial join populacja × buf3000...


Population features: 43,811 × 5


## 4.5 Ruch pieszy — Snowflake (batch, 500 m + 2000 m)

In [6]:
# Optymalizacja: odpytujemy Snowflake tylko dla punktów miejskich
# (buildings_count_3000m >= 2000), pomijając ~97% siatki bez wartości handlowej.

# Identyfikuj kandydatów miejskich już na podstawie cech budynków
urban_ids = set(
    building_features.loc[
        building_features['buildings_count_3000m'] >= 2000, 'location_id'
    ]
)
grid_urban = grid_df[grid_df['location_id'].isin(urban_ids)].reset_index(drop=True)
print(f'Punktów miejskich do odpytania Snowflake: {len(grid_urban):,} / {len(grid_df):,}')

sf = creds['snowflake']
conn = snowflake.connector.connect(
    account=sf['account'], user=sf['user'], password=sf['password'],
    warehouse=sf['warehouse'], database=sf['database'],
    schema=sf['schema'], role=sf['role']
)
cur = conn.cursor()

BATCH_SIZE = 200
footfall_results = []
total_batches = max(1, (len(grid_urban) + BATCH_SIZE - 1) // BATCH_SIZE)
print(f'Wysyłam {total_batches} batchów do Snowflake (active_days_2000m)...')

for i in range(0, len(grid_urban), BATCH_SIZE):
    batch = grid_urban.iloc[i:i+BATCH_SIZE]
    values_rows = ',\n        '.join(
        f"('{row.location_id}', {row.lat}, {row.lng})"
        for _, row in batch.iterrows()
    )
    sql = f'''
    WITH locs AS (
        SELECT column1 AS location_id,
               ST_MAKEPOINT(column3, column2) AS geom
        FROM VALUES {values_rows}
    )
    SELECT
        l.location_id,
        COUNT(DISTINCT DATE(TO_TIMESTAMP_NTZ(t.occured_at))) AS active_days_2000m
    FROM RECRUITMENT_TRACES t
    CROSS JOIN locs l
    WHERE TRY_CAST(t.latitude  AS FLOAT) BETWEEN {lat_min} AND {lat_max}
      AND TRY_CAST(t.longitude AS FLOAT) BETWEEN {lng_min} AND {lng_max}
      AND ST_DISTANCE(
            ST_MAKEPOINT(TRY_CAST(t.longitude AS FLOAT), TRY_CAST(t.latitude AS FLOAT)),
            l.geom
          ) <= 2000
    GROUP BY l.location_id
    '''
    cur.execute(sql)
    footfall_results.extend(cur.fetchall())

    batch_num = i // BATCH_SIZE + 1
    if batch_num % 5 == 0 or batch_num == total_batches:
        print(f'  Batch {batch_num}/{total_batches}')

cur.close()
conn.close()

grid_footfall = pd.DataFrame(
    footfall_results,
    columns=['location_id', 'active_days_2000m']
)
# Punkty miejskie bez żadnego ruchu: active_days_2000m = 0
all_urban_df = pd.DataFrame({'location_id': list(urban_ids)})
grid_footfall = all_urban_df.merge(grid_footfall, on='location_id', how='left').fillna(0)

print(f'Footfall: {len(grid_footfall):,} miejskich punktów')
print(f'  active_days_2000m median = {grid_footfall.active_days_2000m.median():.0f}')


Punktów miejskich do odpytania Snowflake: 35,880 / 43,811


Wysyłam 180 batchów do Snowflake (active_days_2000m)...


  Batch 5/180


  Batch 10/180


  Batch 15/180


...

  Batch 160/180


  Batch 165/180


  Batch 170/180


  Batch 175/180


  Batch 180/180
Footfall: 35,880 miejskich punktów
  active_days_2000m median = 19


## 4.6 Złożenie feature matrix

In [7]:
# Buduj DataFrame ze wszystkimi 93 feature_cols wypełnionymi 0
grid_feat = grid_df[['location_id','lat','lng']].copy()
for col in feature_cols:
    grid_feat[col] = 0.0

# Wklej budynki
merge_cols_b = [col for col in building_features.columns
                if col != 'location_id' and col in feature_cols]
grid_feat = grid_feat.drop(columns=merge_cols_b, errors='ignore')
grid_feat = grid_feat.merge(building_features[['location_id'] + merge_cols_b],
                             on='location_id', how='left')
grid_feat[merge_cols_b] = grid_feat[merge_cols_b].fillna(0)

# Wklej populację
merge_cols_p = [col for col in pop_features.columns
                if col != 'location_id' and col in feature_cols]
grid_feat = grid_feat.drop(columns=merge_cols_p, errors='ignore')
grid_feat = grid_feat.merge(pop_features[['location_id'] + merge_cols_p],
                             on='location_id', how='left')
grid_feat[merge_cols_p] = grid_feat[merge_cols_p].fillna(0)

# Wklej active_days_2000m (Snowflake — tylko miejskie punkty, reszta = 0)
if 'active_days_2000m' in feature_cols and 'active_days_2000m' in grid_footfall.columns:
    grid_feat = grid_feat.drop(columns=['active_days_2000m'], errors='ignore')
    grid_feat = grid_feat.merge(
        grid_footfall[['location_id', 'active_days_2000m']],
        on='location_id', how='left'
    )
    grid_feat['active_days_2000m'] = grid_feat['active_days_2000m'].fillna(0)

# Dist do najbliższego konkurenta
comp_coords_rad = np.radians(competitors[['lat','lng']].values)
grid_coords_rad = np.radians(grid_df[['lat','lng']].values)
bt = BallTree(comp_coords_rad, metric='haversine')
dist_m_arr, _ = bt.query(grid_coords_rad, k=1)
dist_vals = dist_m_arr[:, 0] * 6_371_000
dist_comp_df = pd.DataFrame({'location_id': grid_df['location_id'].values,
                              'dist_nearest_competitor': dist_vals})
grid_feat = grid_feat.drop(columns=['dist_nearest_competitor',
                                     'log_dist_nearest_competitor'], errors='ignore')
grid_feat = grid_feat.merge(dist_comp_df, on='location_id', how='left')
if 'log_dist_nearest_competitor' in feature_cols:
    grid_feat['log_dist_nearest_competitor'] = np.log1p(grid_feat['dist_nearest_competitor'])

# Uzupełnij brakujące cechy zerem i zapewnij kolejność
for c in feature_cols:
    if c not in grid_feat.columns:
        grid_feat[c] = 0.0

# Log1p dla skośnych cech (jak w nb03)
X_grid = grid_feat[feature_cols].copy().clip(lower=0)
X_grid[log_cols] = np.log1p(X_grid[log_cols])

print(f'Feature matrix: {X_grid.shape}')
print(f'NaN: {X_grid.isna().sum().sum()}')


Feature matrix: (43811, 93)
NaN: 0


## 4.7 Predykcja przychodu

In [8]:
raw_pred = pipeline.predict(X_grid.values)
grid_feat['predicted_revenue'] = np.expm1(raw_pred) if log_target else raw_pred
grid_feat['score_pct'] = grid_feat['predicted_revenue'].rank(pct=True)

print(f'Predykcje gotowe dla {len(grid_feat):,} punktów')
print(f'Revenue: min={grid_feat.predicted_revenue.min():,.0f}  '
      f'median={grid_feat.predicted_revenue.median():,.0f}  '
      f'max={grid_feat.predicted_revenue.max():,.0f} PLN')

grid_feat.to_csv('../data/processed/grid_features.csv', index=False)
print('Zapisano grid_features.csv')


Predykcje gotowe dla 43,811 punktów
Revenue: min=169,545  median=189,449  max=467,651 PLN


Zapisano grid_features.csv


## 4.8 Filtrowanie whitespotów

In [9]:
# Wyklucz punkty bliżej niż 500 m od istniejącego sklepu klienta
loc_gdf = gpd.GeoDataFrame(
    locations,
    geometry=gpd.points_from_xy(locations.lng, locations.lat),
    crs=4326
).to_crs(2180)

grid_gdf_2180 = gpd.GeoDataFrame(
    grid_feat,
    geometry=gpd.points_from_xy(grid_feat.lng, grid_feat.lat),
    crs=4326
).to_crs(2180)

min_dist_existing = grid_gdf_2180.geometry.apply(
    lambda p: loc_gdf.geometry.distance(p).min()
)
grid_feat['dist_existing_m'] = min_dist_existing.values

# Filtr miejski: przynajmniej 2000 budynków w 3000 m → punkt w obszarze zabudowanym
URBAN_BUILDINGS_MIN = 2000
urban_mask = (
    (min_dist_existing > 500) &
    (grid_feat.get('buildings_count_3000m', 0) >= URBAN_BUILDINGS_MIN)
)

whitespots = grid_feat[urban_mask].copy()
print(f'Wszystkich punktów siatki: {len(grid_feat):,}')
print(f'  - po filtrze >500 m od sklepu klienta: {(min_dist_existing > 500).sum():,}')
print(f'  - po filtrze miejskim (buildings≥{URBAN_BUILDINGS_MIN}): {len(whitespots):,}')

top20 = whitespots.nlargest(20, 'predicted_revenue').reset_index(drop=True)
print(f'\nTop 20 whitespotów (preview):')
cols_show = ['location_id','lat','lng','predicted_revenue','score_pct',
             'buildings_count_3000m','population_per_km2_3000m',
             'active_days_2000m','dist_nearest_competitor','dist_existing_m']
print(top20[[c for c in cols_show if c in top20.columns]].to_string())


Wszystkich punktów siatki: 43,811
  - po filtrze >500 m od sklepu klienta: 43,514
  - po filtrze miejskim (buildings≥2000): 35,586

Top 20 whitespotów (preview):
   location_id     lat     lng  predicted_revenue  score_pct  buildings_count_3000m  population_per_km2_3000m  active_days_2000m  dist_nearest_competitor  dist_existing_m
0   GRID_40886  50.068  19.956      467650.867054   1.000000                  15476               4615.953131               98.0              1070.391552      1239.178351
1   GRID_40984  50.068  19.960      464854.457926   0.999977                  15007               4347.936206               96.0               943.680552      1148.903535
2   GRID_40983  50.064  19.960      458972.798247   0.999954                  15189               4075.958093               90.0               758.445840       719.248533
3   GRID_40683  50.052  19.948      457713.730171   0.999932                  14586               3546.184339               83.0               613.596427 

## 4.9 Wizualizacja — mapa folium

In [10]:
center_lat = (lat_min + lat_max) / 2
center_lng = (lng_min + lng_max) / 2

m = folium.Map(location=[center_lat, center_lng], zoom_start=10, tiles='CartoDB positron')

# Granica obszaru analizy
folium.GeoJson(
    {'type': 'Feature', 'geometry': area_poly.__geo_interface__, 'properties': {}},
    style_function=lambda x: {'fillColor': 'transparent', 'color': 'navy', 'weight': 2},
    name='Obszar analizy'
).add_to(m)

# Heatmapa — top 5000 punktów miejskich wg score_pct
heatmap_src = (
    grid_feat[grid_feat.get('buildings_count_3000m', 0) >= URBAN_BUILDINGS_MIN]
    .nlargest(5000, 'score_pct')[['lat','lng','score_pct']]
)
fplug.HeatMap(
    heatmap_src.values.tolist(),
    radius=14, blur=10,
    gradient={0.2:'blue', 0.5:'yellow', 0.8:'orange', 1.0:'red'},
    name='Heatmapa potencjału'
).add_to(m)

# Top 20 whitespotów — zielone markery
ws_group = folium.FeatureGroup(name='Top 20 Whitespotów')
for rank, row in top20.fillna(0).iterrows():
    active = int(row.get('active_days_2000m', 0))
    pop    = int(row.get('population_per_km2_3000m', 0))
    comp   = int(row.get('dist_nearest_competitor', 0))
    bld    = int(row.get('buildings_count_3000m', 0))
    folium.CircleMarker(
        [row.lat, row.lng],
        radius=12,
        color='darkgreen', fill=True, fill_color='lime', fill_opacity=0.85,
        popup=folium.Popup(
            f"<b>Whitespot #{rank+1}</b><br>"
            f"Przewidywany przychód: <b>{row.predicted_revenue:,.0f} PLN</b><br>"
            f"Percentyl: {row.score_pct:.1%}<br>"
            f"Lat/Lng: {row.lat:.4f}, {row.lng:.4f}<br>"
            f"Budynki 3 km: {bld:,}<br>"
            f"Gęstość zaludnienia 3 km: {pop:,} os/km²<br>"
            f"Active days 2 km (Snowflake): {active}<br>"
            f"Dist do konkurenta: {comp:,} m<br>"
            f"Dist do wł. sklepu: {int(row.dist_existing_m):,} m",
            max_width=270
        ),
        tooltip=f'#{rank+1} — {row.predicted_revenue:,.0f} PLN'
    ).add_to(ws_group)
ws_group.add_to(m)

# Istniejące sklepy klienta — czerwone
loc_group = folium.FeatureGroup(name='Sklepy klienta')
for _, row in locations.iterrows():
    folium.CircleMarker(
        [row.lat, row.lng],
        radius=7, color='darkred', fill=True, fill_color='red', fill_opacity=0.9,
        popup=f"{row.location_id}<br>Revenue: {row.monthly_revenue:,.0f} PLN",
        tooltip=row.location_id
    ).add_to(loc_group)
loc_group.add_to(m)

# Konkurencja — Żabka niebieska, reszta szara
comp_group = folium.FeatureGroup(name='Konkurencja')
for _, row in competitors.iterrows():
    color = 'blue' if row.brand == 'Żabka' else 'gray'
    folium.CircleMarker(
        [row.lat, row.lng],
        radius=4, color=color, fill=True, fill_opacity=0.5,
        popup=row.brand
    ).add_to(comp_group)
comp_group.add_to(m)

folium.LayerControl().add_to(m)

os.makedirs('../data/processed', exist_ok=True)
m.save('../data/processed/whitespot_map.html')
print('Mapa zapisana → data/processed/whitespot_map.html')
m


Mapa zapisana → data/processed/whitespot_map.html


## 4.10 Dystrybucja whitespotów per dzielnica

In [11]:
districts_gdf = gpd.GeoDataFrame(
    districts_df,
    geometry=gpd.GeoSeries.from_wkt(districts_df['geometry']),
    crs=4326
)

ws_gdf = gpd.GeoDataFrame(
    whitespots,
    geometry=gpd.points_from_xy(whitespots.lng, whitespots.lat),
    crs=4326
)

ws_dist = gpd.sjoin(
    ws_gdf[['location_id','predicted_revenue','score_pct','geometry']],
    districts_gdf[['name','geometry']],
    how='left', predicate='within'
)

district_summary = ws_dist.groupby('name').agg(
    n_whitespots   =('location_id',       'count'),
    avg_revenue    =('predicted_revenue', 'mean'),
    top_revenue    =('predicted_revenue', 'max'),
    avg_score_pct  =('score_pct',         'mean'),
).sort_values('avg_revenue', ascending=False)

print('Top dzielnice wg średniego przewidywanego przychodu whitespotów (filtr miejski):')
print(district_summary.round(0).to_string())

# Lokalizacje top20 z dzielnicą
top20_with_dist = gpd.sjoin(
    gpd.GeoDataFrame(top20, geometry=gpd.points_from_xy(top20.lng, top20.lat), crs=4326),
    districts_gdf[['name','geometry']],
    how='left', predicate='within'
)
print('\nTop 20 z dzielnicą:')
print(top20_with_dist[['location_id','predicted_revenue','name',
                         'buildings_count_3000m','population_per_km2_3000m',
                         'active_days_2000m']].to_string())


Top dzielnice wg średniego przewidywanego przychodu whitespotów (filtr miejski):
                   n_whitespots  avg_revenue  top_revenue  avg_score_pct
name                                                                    
Katowice                   1034     275834.0     413613.0            1.0
Kraków                     3030     230872.0     467651.0            1.0
Sosnowiec                  1328     227142.0     303271.0            1.0
Bytom                      3011     223858.0     301458.0            1.0
powiat wielicki             671     212358.0     300982.0            1.0
Gliwice                    3589     211537.0     323511.0            1.0
Tychy                      2254     207199.0     282067.0            1.0
Oświęcim                   2238     204895.0     284869.0            1.0
Dąbrowa Górnicza           2179     201920.0     302361.0            0.0
Jaworzno                   1840     200466.0     253295.0            1.0
Rybnik                     1095     199005.

## 4.11 Wnioski i rekomendacje

Poniżej wypełniane automatycznie po uruchomieniu notebook'a — top lokalizacje z ich kluczowymi metrykami.


In [12]:
print('=== TOP 5 REKOMENDOWANYCH LOKALIZACJI ===')
top5 = top20_with_dist.head(5)
for rank, (_, row) in enumerate(top5.iterrows(), 1):
    district = row.get('name', 'N/A')
    rev = row['predicted_revenue']
    pop = row.get('population_per_km2_3000m', 0)
    bld = row.get('buildings_count_3000m', 0)
    active = row.get('active_days_2000m', 0)
    dist_comp = row.get('dist_nearest_competitor', 0)
    dist_own  = row.get('dist_existing_m', 0)
    print(f'\n#{rank}  [{district}]  Lat={row.lat:.4f}, Lng={row.lng:.4f}')
    print(f'     Przewidywany przychód:  {rev:,.0f} PLN/mies.')
    print(f'     Gęstość zaludnienia 3km: {int(pop):,} os/km²')
    print(f'     Budynki 3km:            {int(bld):,}')
    print(f'     Active days 2km:        {int(active)} dni')
    print(f'     Dist. do konkurenta:    {int(dist_comp):,} m')
    print(f'     Dist. do własnego skl.: {int(dist_own):,} m')

print('\n=== UWAGI METODYCZNE ===')
print(f'Model: {saved["best_name"]} (k={saved["best_k"]} cech)')
print(f'CV R² = {saved["cv_R2_mean"]:.3f} ± {saved["cv_R2_std"]:.3f}  |  RMSE = {saved["cv_RMSE_PLN"]:,.0f} PLN')
print('Zbiór treningowy n=49 — przedziały ufności szerokie.')
print('Dane mobilne z 2020 (COVID) — active_days może być zaniżony.')
print('Model nie uwzględnia czynszów, dostępności lokalu, ekspozycji witryny.')


=== TOP 5 REKOMENDOWANYCH LOKALIZACJI ===

#1  [Kraków]  Lat=50.0680, Lng=19.9560
     Przewidywany przychód:  467,651 PLN/mies.
     Gęstość zaludnienia 3km: 4,615 os/km²
     Budynki 3km:            15,476
     Active days 2km:        98 dni
     Dist. do konkurenta:    1,070 m
     Dist. do własnego skl.: 1,239 m

#2  [Kraków]  Lat=50.0680, Lng=19.9600
     Przewidywany przychód:  464,854 PLN/mies.
     Gęstość zaludnienia 3km: 4,347 os/km²
     Budynki 3km:            15,007
     Active days 2km:        96 dni
     Dist. do konkurenta:    943 m
     Dist. do własnego skl.: 1,148 m

#3  [Kraków]  Lat=50.0640, Lng=19.9600
     Przewidywany przychód:  458,973 PLN/mies.
     Gęstość zaludnienia 3km: 4,075 os/km²
     Budynki 3km:            15,189
     Active days 2km:        90 dni
     Dist. do konkurenta:    758 m
     Dist. do własnego skl.: 719 m

#4  [Kraków]  Lat=50.0520, Lng=19.9480
     Przewidywany przychód:  457,714 PLN/mies.
     Gęstość zaludnienia 3km: 3,546 os/km²
     B